# Rabbit Hole Finder: FULL build on Kaggle (2× T4)

Builds the FULL dataset with the same Dagster pipeline as `make pipeline`, minus the final
Postgres load (that happens on your machine). Output: `/kaggle/working/rabbithole_full/`.

**Settings before running:** Accelerator **GPU T4 ×2**, Internet **on** (needs a phone-verified
account). Then **Save Version → Save & Run All (Commit)** so it runs in the background (~2–4 h).

What runs where:
- clickstream download + parse, FULL subset, link and label edges: CPU
- article text: streamed from the Hugging Face `wikimedia/wikipedia` dump (no big single download)
- article + chunk embeddings: **both GPUs**, fp16, one encoder process per GPU
- Leiden sweep, rabbit-holeness, pruned graph, UMAP map: CPU

In [ ]:
# Settings: the only cell you might edit.
FULL_MIN_VIEWS = 1000        # >=1000 views in the feature month ~ 654k articles (spec default)
CLICKSTREAM_MONTHS = "2026-07,2026-08,2026-09"   # same months as the local DEV build
MAX_CHUNKS_PER_ARTICLE = 8
LEIDEN_RESOLUTIONS = [1, 2, 3, 4, 6, 8, 12, 20, 30, 50, 80]  # FULL needs finer settings than DEV
REPO = "https://github.com/Siddhartha121220/RabbitHole_Finder.git"
BRANCH = "main"

In [ ]:
!nvidia-smi -L
!df -h /kaggle/working /kaggle/temp /tmp 2>/dev/null
!python --version

In [ ]:
import os, subprocess, sys, tomllib

subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, "/kaggle/working/src"], check=True)
deps = tomllib.load(open("/kaggle/working/src/backend/pyproject.toml", "rb"))["project"]["dependencies"]
# Keep Kaggle's CUDA build of torch; skip serving-only packages.
skip = ("torch", "fastapi", "uvicorn", "anthropic", "mlflow", "dagster-webserver")
deps = [d for d in deps if not d.split(">")[0].split("=")[0].strip().startswith(skip)]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *deps], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--ignore-requires-python",
                "-e", "/kaggle/working/src/backend"], check=True)

import torch
print("torch", torch.__version__, "GPUs:", torch.cuda.device_count())
assert torch.cuda.device_count() >= 1, "Turn on the GPU accelerator (T4 x2)"

In [ ]:
import json
data_dir = "/kaggle/temp/data" if os.path.isdir("/kaggle/temp") else "/tmp/rabbithole-data"
gpus = [f"cuda:{i}" for i in range(torch.cuda.device_count())]
env = {
    "DATASET_MODE": "full",
    "DATA_DIR": data_dir,
    "FULL_MIN_VIEWS": str(FULL_MIN_VIEWS),
    "CLICKSTREAM_MONTHS": CLICKSTREAM_MONTHS,
    "MAX_CHUNKS_PER_ARTICLE": str(MAX_CHUNKS_PER_ARTICLE),
    "LEIDEN_RESOLUTIONS": json.dumps(LEIDEN_RESOLUTIONS),
    "EMBED_DEVICES": json.dumps(gpus),
    "EMBED_BATCH_SIZE": "256",
    "EMBED_SHARD_ROWS": "100000",
    "DAGSTER_HOME": "/kaggle/temp/dagster" if os.path.isdir("/kaggle/temp") else "/tmp/dagster",
    "TOKENIZERS_PARALLELISM": "false",
}
os.environ.update(env)
os.makedirs(env["DATA_DIR"], exist_ok=True)
os.makedirs(env["DAGSTER_HOME"], exist_ok=True)
print(json.dumps(env, indent=1))

In [ ]:
# Everything except the Postgres load (13 assets + their data checks), one step at a time
# so the RAM-heavy steps (UMAP, Leiden, embeddings) never overlap on Kaggle's ~29 GB.
open("/kaggle/working/run.yaml", "w").write("execution:\n  config:\n    multiprocess:\n      max_concurrent: 1\n")
!cd /kaggle/working/src && dagster asset materialize -m rabbithole.pipelines.definitions \
    -c /kaggle/working/run.yaml \
    --select '*map_coords,*rh_scores,*chunk_embeddings,*label_edges,*edges_pruned'


In [ ]:
# Bundle what serving, LTR and evaluation need (article bodies are left behind).
!cd /kaggle/working/src && rabbithole pack --out /kaggle/working/rabbithole_full
!cat /kaggle/working/rabbithole_full/manifest.json
!du -sh /kaggle/working/rabbithole_full
# Free the output folder of everything else, so only the bundle is saved.
!rm -rf /kaggle/working/src /kaggle/working/run.yaml

## Next, on your machine
```bash
pip install kaggle   # once; put your API token in ~/.kaggle/kaggle.json
kaggle kernels output <your-username>/<this-notebook-slug> -p ~/Downloads/rabbithole_full
cd rabbithole
uv --directory backend run rabbithole unpack ~/Downloads/rabbithole_full/rabbithole_full
DATASET_MODE=full make load      # Postgres (streams chunks; ~20–40 min)
```